# Clase 07 · Implementación · De los documentos al índice
**Inteligencia Artificial Generativa y Sistemas Inteligentes · EAM 2026-II**

Sebastián Cruz Cardona    
Código: 240220222010

En esta práctica prepararemos dos documentos, compararemos dos formas de fragmentarlos y guardaremos un índice para volver a consultarlo. Trabajaremos en Colab, con el mismo modelo de embeddings de la clase anterior. La primera ejecución necesita Internet para instalar la biblioteca y descargar el modelo; podemos usar CPU.

Ejecuten las celdas en orden. Los procedimientos del ejemplo son **ficticios**. La consulta se escribe en el punto **5**.

## 1. Preparación
Instalamos la biblioteca y cargamos el modelo. Comprobamos su límite de entrada para evitar que el texto se recorte al generar los embeddings.

In [1]:
%pip -q install sentence-transformers==6.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 27.1 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import re, json, hashlib, zipfile
import numpy as np
from sentence_transformers import SentenceTransformer
from huggingface_hub import model_info

# Modelo de embeddings
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# Versión del modelo
REVISION = model_info(MODELO).sha
# Cargar el modelo
modelo = SentenceTransformer(MODELO, revision=REVISION)
# Tokenizador del modelo
tokenizador = modelo.tokenizer

# Límites de Tokens
MAX_TOKENS = 95       # Presupuesto del fragmento completo, incluido el encabezado.
SOLAPAMIENTO = 17    # Tokens repetidos entre ventanas de un mismo bloque.

def contar_tokens(texto):
    """
    Recibe un texto y retorna un número entero con la cantidad real de
    tokens que el modelo procesará.

    - Añade los tokens especiales que el modelo necesita internamente,
      como marcadores de inicio o final.

    - Prohíbe recortar el texto automáticamente para detectar si un texto
      supera el límite, en lugar de perder información silenciosamente.
    """
    return len(tokenizador.encode(texto, add_special_tokens=True, truncation=False))

print("Límite del modelo:", modelo.max_seq_length, "tokens")
print("Presupuesto de la práctica:", MAX_TOKENS, "tokens")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Límite del modelo: 128 tokens
Presupuesto de la práctica: 95 tokens


## 2. Los documentos del corpus
Esta celda crea dos archivos Markdown en la sesión. Después los leeremos desde la carpeta, como haremos con las fuentes del proyecto.

In [10]:
# Crear directorio con ruta relativa al entorno de ejecución actual
carpeta = Path("corpus_clase07")
carpeta.mkdir(exist_ok=True)

# Diccionario de documentos a transformar en archivos
fuentes = {
    "constitucion_politica.md": """# CONSTITUCION POLITICA DE COLOMBIA

## Artículo 1

Colombia   es un Estado social de derecho, organizado en forma de República unitaria, descentralizada, con autonomía de sus entidades territoriales, democrática, participativa y pluralista, fundada en el respeto de la dignidad humana, en el trabajo y la solidaridad de las personas que la integran y en la prevalencia del interés general.

## Artículo 2

Son fines esenciales del Estado: servir a la comunidad, promover la prosperidad general y garantizar la efectividad de los principios, derechos y deberes consagrados en la Constitución; facilitar la participación de todos en las decisiones que los afectan y en la vida económica, política, administrativa y cultural de la Nación; defender la independencia nacional, mantener la integridad territorial y asegurar la convivencia pacifica y la vigencia de un orden justo.


Las autoridades de la República están instituidas para proteger a todas las personas
residentes en Colombia, en su vida, honra, bienes, creencias, y demás derechos y
libertades, y para asegurar el cumplimiento de los deberes sociales del Estado y de los particulares.

## Artículo 3

La soberanía reside exclusivamente en el pueblo, del cual emana el poder público. El pueblo la ejerce en forma directa o por medio de sus representantes, en los términos que la Constitución establece.

## Artículo 4

La Constitución es norma de normas. En todo caso de incompatibilidad entre la Constitución y la ley u otra norma jurídica, se aplicarán las disposiciones
constitucionales.

Es deber   de los nacionales y de los extranjeros en Colombia acatar la Constitución y las leyes, y respetar y obedecer a las autoridades.

## Artículo 5

El Estado reconoce, sin discriminación alguna, la primacía de los derechos
inalienables de la persona y ampara a la familia como institución básica de la sociedad.

## Artículo 6

Los particulares sólo son responsables ante las autoridades por infringir la
Constitución y las leyes. Los servidores públicos lo son por la misma causa y por
omisión o extralimitación en el ejercicio de sus funciones.

## Artículo 7

El Estado reconoce y protege la diversidad étnica y cultural de la Nación
colombiana.

## Artículo 8

Es obligación del Estado y de las personas proteger las riquezas culturales y
naturales de la Nación.

## Artículo 9

Las relaciones exteriores del Estado se fundamentan en la soberanía
nacional, en el respeto a la autodeterminación de los pueblos y en el reconocimiento de
los principios del derecho internacional aceptados por Colombia.
De igual manera, la política exterior de Colombia se orientará hacia la
integración latinoamericana y del Caribe.

## Artículo 10

El castellano es el idioma oficial de Colombia. Las lenguas y dialectos de los
grupos étnicos son también oficiales en sus territorios. La enseñanza que se imparta
en las comunidades con tradiciones lingüísticas propias será bilingüe.

## Artículo 11

El derecho a la vida es inviolable. No habrá pena de muerte.

## Artículo 12

Nadie será sometido a desaparición forzada, a torturas ni a tratos o penas
crueles, inhumanos o degradantes.

## Artículo 13

Todas las personas nacen libres e iguales ante la ley, recibirán la misma
protección y trato de las autoridades y gozarán de los mismos derechos, libertades y
oportunidades sin ninguna discriminación por razones de sexo, raza, origen nacional o
familiar, lengua, religión, opinión política o filosófica.
El Estado promoverá las condiciones para que la igualdad sea real y efectiva y adoptara
medidas en favor de grupos discriminados o marginados.
El Estado protegerá especialmente a aquellas personas que por su condición
económica, física o mental, se encuentren en circunstancia de debilidad manifiesta y
sancionará los abusos o maltratos que contra ellas se cometan.

## Artículo 14

Toda persona tiene derecho al reconocimiento de su personalidad jurídica.

## Artículo 15

Todas las personas tienen derecho a su intimidad personal y familiar y a su
buen nombre, y el Estado debe respetarlos y hacerlos respetar. De igual modo, tienen
derecho a conocer, actualizar y rectificar las informaciones que se hayan recogido sobre
ellas en bancos de datos y en archivos de entidades públicas y privadas.
En la recolección, tratamiento y circulación de datos se respetarán la libertad y demás
garantías consagradas en la Constitución.
La correspondencia y demás formas de comunicación privada son inviolables. Sólo
pueden ser interceptadas o registradas mediante orden judicial, en los casos y con las
formalidades que establezca la ley.
Para efectos tributarios o judiciales y para los casos de inspección, vigilancia e
intervención del Estado podrá exigirse la presentación de libros de contabilidad y demás
documentos privados, en los términos que señale la ley.

## Artículo 16

Todas las personas tienen derecho al libre desarrollo de su personalidad
sin más limitaciones que las que imponen los derechos de los demás y el orden jurídico.

## Artículo 17

Se prohiben la esclavitud, la servidumbre y la trata de seres humanos en
todas sus formas.

## Artículo 18

Se garantiza la libertad de conciencia. Nadie será molestado por razón de sus
convicciones o creencias ni compelido a revelarlas ni obligado a actuar contra su
conciencia.

## Artículo 19

Se garantiza la libertad de cultos. Toda persona tiene derecho a profesar
libremente su religión y a difundirla en forma individual o colectiva.
Todas las confesiones religiosas e iglesias son igualmente libres ante la ley.

## Artículo 20

Se garantiza a toda persona la libertad de expresar y difundir su
pensamiento y opiniones, la de informar y recibir información veraz e imparcial, y la de
fundar medios masivos de comunicación.
Estos son libres y tienen responsabilidad social. Se garantiza el derecho a la
rectificación en condiciones de equidad. No habrá censura.

## Artículo 21

Se garantiza el derecho a la honra. La ley señalará la forma de su
protección.

## Artículo 22

La paz es un derecho y un deber de obligatorio cumplimiento.

## Artículo 23

Toda persona tiene derecho a presentar peticiones respetuosas a las
autoridades por motivos de interés general o particular y a obtener pronta resolución. El
legislador podrá reglamentar su ejercicio ante organizaciones privadas para garantizar
los derechos fundamentales.

## Artículo 24

Todo colombiano, con las limitaciones que establezca la ley, tiene derecho
a circular libremente por el territorio nacional, a entrar y salir de él, y a permanecer y
residenciarse en Colombia.

## Artículo 25

El trabajo es un derecho y una obligación social y goza, en todas sus
modalidades, de la especial protección del Estado. Toda persona tiene derecho a un
trabajo en condiciones dignas y justas.

## Artículo 26

Toda persona es libre de escoger profesión u oficio. La ley podrá exigir
títulos de idoneidad. Las autoridades competentes inspeccionaran y vigilaran el ejercicio
de las profesiones. Las ocupaciones, artes y oficios que no exijan formación académica
son de libre ejercicio, salvo aquellas que impliquen un riesgo social.
Las profesiones legalmente reconocidas pueden organizarse en colegios. La estructura
interna y el funcionamiento de estos deberán ser democráticos.
La ley podrá asignarles funciones públicas y establecer los debidos controles.

## Artículo 27

El Estado garantiza las libertades de enseñanza, aprendizaje, investigación
y cátedra.

## Artículo 28

Toda persona es libre. Nadie puede ser molestado en su persona o familia,
ni reducido a prisión o arresto, ni detenido, ni su domicilio registrado, sino en virtud de
mandamiento escrito de autoridad judicial competente, con las formalidades legales y
por motivo previamente definido en la ley.
La persona detenida preventivamente será puesta a disposición del juez competente
dentro de las treinta y seis horas siguientes, para que éste adopte la decisión
correspondiente en el término que establezca la ley. En ningún caso podrá haber
detención, prisión ni arresto por deudas, ni penas y medidas de seguridad
imprescriptibles.

## Artículo 29

El debido proceso se aplicará a toda clase de actuaciones judiciales y
administrativas.
Nadie podrá ser juzgado sino conforme a leyes preexistentes al acto que se le imputa,
ante juez o tribunal competente y con observancia de la plenitud de las formas propias
de cada juicio.
En materia penal, la ley permisiva o favorable, aun cuando sea posterior, se aplicará de
preferencia a la restrictiva o desfavorable.
Toda persona se presume inocente mientras no se la haya declarado judicialmente
culpable. Quien sea sindicado tiene derecho a la defensa y a la asistencia de un
abogado escogido por el, o de oficio, durante la investigación y el juzgamiento; a un
debido proceso publico sin dilaciones injustificadas; a presentar pruebas y a controvertir
las que se alleguen en su contra; a impugnar la sentencia condenatoria, y a no ser
juzgado dos veces por el mismo hecho.
Es nula, de pleno derecho, la prueba obtenida con violación del debido proceso.

## Artículo 30

Quien estuviere privado de su libertad, y creyere estarlo ilegalmente, tiene
derecho a invocar ante cualquier autoridad judicial, en todo tiempo, por si o por
interpuesta persona, el Habeas Corpus, el cual debe resolverse en el término de treinta
y seis horas.

## Artículo 31

Toda sentencia judicial podrá ser apelada o consultada, salvo las
excepciones que consagre la ley.
El superior no podrá agravar la pena impuesta cuando el condenado sea apelante
único.

## Artículo 32

El delincuente sorprendido en flagrancia podrá ser aprehendido y llevado
ante el juez por cualquier persona. Si los agentes de la autoridad lo persiguieren y se
refugiare en su propio domicilio, podrán penetrar en él, para el acto de la aprehensión;
si se acogiere a domicilio ajeno, deberá preceder requerimiento al morador.

## Artículo 33

Nadie podrá ser obligado a declarar contra si mismo o contra su cónyuge,
compañero permanente o parientes dentro del cuarto grado de consanguinidad,
segundo de afinidad o primero civil.

## Artículo 34

Se prohiben las penas de destierro, prisión perpetua y confiscación.
No obstante, por sentencia judicial, se declarará extinguido el dominio sobre los bienes
adquiridos mediante enriquecimiento ilícito, en perjuicio del Tesoro Público o con grave
deterioro de la moral social.

## Artículo 35

Se prohibe la extradición de colombianos por nacimiento. No se concederá
la extradición de extranjeros por delitos políticos o de opinión.
Los colombianos que hayan cometido delitos en el exterior, considerados como tales en
la legislación nacional, serán procesados y juzgados en Colombia.

## Artículo 36

Se reconoce el derecho de asilo en los términos previstos en la ley.

## Artículo 37

Toda parte del pueblo puede reunirse y manifestarse pública y
pacíficamente. Sólo la ley podrá establecer de manera expresa los casos en los cuales
se podrá limitar el ejercicio de este derecho.

## Artículo 38

Se garantiza el derecho de libre asociación para el desarrollo de las
distintas actividades que las personas realizan en sociedad.

## Artículo 39

Los trabajadores y empleadores tienen derecho a constituir sindicatos o
asociaciones, sin intervención del Estado. Su reconocimiento jurídico se producirá con
la simple inscripción del acta de constitución.
La estructura interna y el funcionamiento de los sindicatos y organizaciones sociales y
gremiales se sujetarán al orden legal y a los principios democráticos.
La cancelación o la suspensión de la personería jurídica sólo procede por vía judicial.
Se reconoce a los representantes sindicales el fuero y las demás garantías necesarias
para el cumplimiento de su gestión.
No gozan del derecho de asociación sindical los miembros de la Fuerza Pública.

## Artículo 40

Todo ciudadano tiene derecho a participar en la conformación, ejercicio y
control del poder político. Para hacer efectivo este derecho puede:
1. Elegir y ser elegido.
2. Tomar parte en elecciones, plebiscitos, referendos, consultas populares y otras
formas de participación democrática.
3. Constituir partidos, movimientos y agrupaciones políticas sin limitación alguna: formar
parte de ellos libremente y difundir sus ideas y programas.
4. Revocar el mandato de los elegidos en los casos y en la forma que establecen la
Constitución y la ley.
5. Tener iniciativa en las corporaciones públicas.
6. Interponer acciones públicas en defensa de la Constitución y de la ley.
7. Acceder al desempeño de funciones y cargos públicos, salvo los colombianos, por
nacimiento o por adopción, que tengan doble nacionalidad. La ley reglamentará esta
excepción y determinará los casos a los cuales ha de aplicarse.
Las autoridades garantizarán la adecuada y efectiva participación de la mujer en los
niveles decisorios de la Administración Pública.

## Artículo 41

En todas las instituciones de educación, oficiales o privadas, serán
obligatorios el estudio de la Constitución y la Instrucción Cívica. Así mismo se
fomentarán prácticas democráticas para el aprendizaje de los principios y valores de
la participación ciudadana. El Estado divulgará la Constitución.
""",
    "guia_uso.md": """# Guía de uso del asistente normativo

## Interacción con  el asistente


El   sistema funciona mediante una interfaz de chat en la que el usuario puede formular preguntas relacionadas con información y procedimientos del ámbito jurídico. Para realizar una consulta, ingrese a la interfaz de entrada y escriba la solicitud de forma clara, proporcionando los datos necesarios según el tipo de consulta.

El asistente analiza la solicitud para identificar su categoría, tipo de consulta y la información disponible. Cuando los datos proporcionados no son suficientes para atender la solicitud, puede solicitar información adicional antes de continuar.

## Consulta del estado de un proceso

Si requiere consultar el estado de un proceso judicial, incluya el código del proceso en la solicitud. El código debe estar conformado por la letra P, seguida de un guion y 11 dígitos, por ejemplo: P-20210010001.

Una consulta puede formularse directamente en el chat, por ejemplo: "¿Cómo va el proceso P-20210010001?". Si se proporciona un código válido, el sistema puede consultar el registro correspondiente y devolver la información disponible sobre su estado.

Si se solicita el estado de un proceso sin proporcionar su código, el sistema solicitará el código del proceso antes de realizar la consulta.

La consulta de procesos es de solo lectura. El sistema no permite crear, modificar, actualizar, alterar o eliminar registros de procesos mediante esta interfaz.

## Información y alcance

El asistente utiliza la información proporcionada en la solicitud para determinar cómo atenderla. No debe asumirse que una respuesta contiene información que no haya sido proporcionada o recuperada mediante las fuentes disponibles.

Las solicitudes que se encuentren fuera del alcance del sistema pueden ser rechazadas o requerir revisión humana. Cuando una solicitud necesite información adicional, el asistente indicará los datos que deben proporcionarse para continuar.
""",
}

# Construir la ruta y guardar el archivo con su respectivo texto
for nombre, contenido in fuentes.items():
    # Ej. corpus_clase07/constancias.md
    (carpeta / nombre).write_text(contenido, encoding="utf-8")

# Mostrar los nombres de los archivos creados
print("Archivos:", ", ".join(fuentes))

Archivos: constitucion_politica.md, guia_uso.md


## 3. Lectura y limpieza
Conservamos archivo, título, versión y huella del original. Reducimos espacios y líneas repetidas; mantenemos párrafos, negaciones y condiciones. Lean la muestra y compárenla con el documento anterior.

In [11]:
# Lista de diccionarios para guardar metadatos y texto limpio de cada archivo
documentos = []

# Recorrer en orden alfabético todos los archivos .md presentes en la carpeta
for ruta in sorted(carpeta.glob("*.md")):
    # Archivo original
    original = ruta.read_text(encoding="utf-8")

    # Limpiar dobles espacios en blanco o tabs, reemplazar por un solo espacio
    limpio = re.sub(r"[ \t]+", " ", original)

    # Reemplazar 3 o más saltos de línea por 2 saltos de línea (separación
    # normal entre párrafos) y eliminar espacios vacíos al inicio y final
    limpio = re.sub(r"\n{3,}", "\n\n", limpio).strip()

    # Guardar metadatos del documento (con identidad y procedencia)
    documentos.append({
        "document_id": ruta.stem, # Nombre del archivo sin extensión
        "source": ruta.name, # Nombre completo con extensión
        "title": limpio.splitlines()[0].lstrip("# "), # Título quitando el # del md
        "version": "1.0-implementacion",
        "source_hash": hashlib.sha256(
            original.encode("utf-8")
        ).hexdigest(), #  Hash del contenido original del archivo para versionado
        "texto": limpio,
    })

    # Mostrar el nombre del archivo, tamaño original y tamaño limpio en caracteres
    print(f"{ruta.name}: {len(original)} → {len(limpio)} caracteres")

# Muestra: primer documento procesado
print("\nMuestra del texto limpio:\n", documentos[0]["texto"][:450])

constitucion_politica.md: 13156 → 13150 caracteres
guia_uso.md: 1959 → 1954 caracteres

Muestra del texto limpio:
 # CONSTITUCION POLITICA DE COLOMBIA

## Artículo 1

Colombia es un Estado social de derecho, organizado en forma de República unitaria, descentralizada, con autonomía de sus entidades territoriales, democrática, participativa y pluralista, fundada en el respeto de la dignidad humana, en el trabajo y la solidaridad de las personas que la integran y en la prevalencia del interés general.

## Artículo 2

Son fines esenciales del Estado: servir a la 


## 4. Dos estrategias de fragmentación
**Ventanas:** recorremos cada sección por tokens, con solapamiento. **Párrafos:** conservamos sus límites y subdividimos solo los párrafos que exceden el presupuesto. En la segunda estrategia no repetimos texto entre párrafos; sí entre ventanas de un párrafo largo.

Ambas incluyen título y sección en el texto vectorizado. Conservamos el mismo modelo, presupuesto y medida de búsqueda. Revisen los tamaños y el final de los fragmentos: ¿alguna condición quedó separada de su consecuencia?

In [12]:
def ventanas(texto, limite):
    """
    Divide un texto en ventanas de tokens con solapamiento entre fragmentos.

    Args:
        texto: Texto que se desea fragmentar.
        limite: Máximo de tokens permitidos por ventana.

    Yields:
        Fragmentos de texto consecutivos, con un solapamiento de
        `SOLAPAMIENTO` tokens entre ventanas.
    """
    # Dividir el texto en tokens y devolver posición de inicio y final para c/u
    offsets = tokenizador(
        texto,
        add_special_tokens=False, # Sin tokens especiales, solo fragmentar contenido
        return_offsets_mapping=True # Para recuperar texto real usando posiciones
    )["offset_mapping"]

    # Validar tamaño de la ventana para garantizar avance en el texto
    if limite <= SOLAPAMIENTO:
        raise ValueError("El encabezado deja muy poco espacio para el contenido.")

    # Recorrer los tokens con salto = límite (95) - Solapamiento (17) = 78
    # Esto garantiza el tamaño de cada ventana y el solapamiento
    for inicio in range(0, len(offsets), limite - SOLAPAMIENTO):
        # Fin de la ventana. Evitar acceder a tokens que no existen en la última ventana.
        fin = min(inicio + limite, len(offsets))
        # Generar una ventana cada vez. Extraer desde el primer hasta el último caracter del token
        yield texto[offsets[inicio][0]:offsets[fin - 1][1]].strip()

        # Detener el for al alcanzar el último token
        if fin == len(offsets):
            break

def fragmentar(estrategia):
    """
    Fragmenta los documentos según la estrategia indicada y genera los
    registros necesarios para su posterior indexación.

    Args:
        estrategia: Estrategia de fragmentación, `ventanas` o `parrafos`.

    Returns:
        Lista de diccionarios de fragmentos con sus metadatos, sección, número
        de tokens y contenido.
    """
    # Lista de diccionarios para almacenar los fragmentos y sus metadatos
    registros = []

    # Procesar cada documento leído y limpiado
    for documento in documentos:

        # Enumerar desde 1 todos los títulos de nivel 2 (##) o títulos de sección
        for numero, bloque in enumerate(re.split(r"(?m)^## ", documento["texto"])[1:], 1):
            # seccion: título de la sección, cuerpo: texto de la sección
            seccion, cuerpo = bloque.split("\n", 1) # Solo una división en \n

            # encabezado: titulo_documento / titulo_seccion
            encabezado = documento["title"] + " / " + seccion + "\n"

            # Tokens disponibles luego de encabezado y 2 de margen
            limite = MAX_TOKENS - contar_tokens(encabezado) - 2

            # Qué se va a fragmentar según la estrategia
            partes = [cuerpo] if estrategia == "ventanas" else re.split(r"\n\s*\n", cuerpo.strip())

            # Procesar cada bloque (ventanas) o párrafo (párrafos)
            for parte in partes:
                # Fragmentar bloque si es necesario. Un párrafo muy largo también
                # Se divide en ventanas
                for texto in ventanas(parte.strip(), limite):
                    # Este es el fragmento o chunk
                    contenido = encabezado + texto

                    # Contar tokens del fragmento completo (encabezado + texto)
                    tokens = contar_tokens(contenido)

                    # Validar la cantidad de tokens del fragmento contra el menor entre
                    # esos dos, en ese caso MAX_TOKENS que es 95
                    if tokens > min(MAX_TOKENS, modelo.max_seq_length):
                        # Lanzar excepción en lugar de truncar texto en silencio
                        raise ValueError("Un fragmento supera el límite: ajuste el presupuesto.")

                    # Guardar fragmento y sus metadatos
                    registros.append({
                        **{ # Metadatos como clave: valor para el diccionario
                            k: documento[k]
                            for k in [
                                "document_id",
                                "source",
                                "title",
                                "version",
                                "source_hash"
                            ]
                        },
                        # Identificador único del fragmento.
                        # documento:versión:estrategia:número del fragmento
                        # ej. acceso:1.0-didactica:ventanas:0
                        "chunk_id": f"{documento['document_id']}:{documento['version']}:{estrategia}:{len(registros)}",
                        "section": seccion, # Título de la sección
                        "section_number": numero, # Número de la sección
                        "tokens": tokens, # Cantidad de tokens
                        "texto": contenido, # Fragmento completo (encabezado + texto)
                    })

    # Devolver todos los fragmentos generados
    return registros

# Ejecutar ambas estrategias. El diccionario tendrá dos claves
# ventas y parrafos. Cada clave tiene su lista de fragmentos resultantes
fragmentos = {
    nombre: fragmentar(nombre)
    for nombre in ["ventanas", "parrafos"]
}

# Recorrer el resultado de ambas estrategias
for nombre, registros in fragmentos.items():
    # Nombre_Estrategia: cantidad_de_fragmentos fragmentos
    # máximo cantidad_de_tokens_máxima_encontrada tokens
    print(
        f"\n{nombre}: {len(registros)} fragmentos; "
        f"máximo {max(r['tokens'] for r in registros)} tokens"
    )

    # Inspeccionar los dos primeros fragmentos para cada estrategia
    # Muestra el contenido para comprobar visualmente si el corte conserva el significado.
    for registro in registros[:2]:
        # Metadatos
        print(registro["chunk_id"], "|", registro["source"], "|", registro["tokens"], "tokens")
        # Fragmento completo (encabezado + texto)
        print(registro["texto"], "\n")


ventanas: 63 fragmentos; máximo 93 tokens
constitucion_politica:1.0-implementacion:ventanas:0 | constitucion_politica.md | 85 tokens
CONSTITUCION POLITICA DE COLOMBIA / Artículo 1
Colombia es un Estado social de derecho, organizado en forma de República unitaria, descentralizada, con autonomía de sus entidades territoriales, democrática, participativa y pluralista, fundada en el respeto de la dignidad humana, en el trabajo y la solidaridad de las personas que la integran y en la prevalencia del interés general. 

constitucion_politica:1.0-implementacion:ventanas:1 | constitucion_politica.md | 93 tokens
CONSTITUCION POLITICA DE COLOMBIA / Artículo 2
Son fines esenciales del Estado: servir a la comunidad, promover la prosperidad general y garantizar la efectividad de los principios, derechos y deberes consagrados en la Constitución; facilitar la participación de todos en las decisiones que los afectan y en la vida económica, política, administrativa y cultural de la Nación; defender la 

## 5. Consulta del usuario y comparación
Calculamos los embeddings de ambos conjuntos una vez. Después escribimos la pregunta en **CONSULTA** y comparamos los tres primeros resultados. Cada resultado conserva el archivo y la sección; vuelvan al original para comprobar su contenido. La similitud coseno ordena candidatos, no mide certeza.

In [13]:
# Calcular los embeddings del corpus.
# El diccionario matrices contine una matriz de vectores para cada estrategia.
# Cada fila de cada matriz representa un fagmento (384 dimensiones - columnas)
matrices = {
    nombre: modelo.encode(
        [r["texto"] for r in registros],
        normalize_embeddings=True, # Normalizar cada vector para longitud 1
        show_progress_bar=False
    )
    for nombre, registros in fragmentos.items()
}

# Función de búsqueda
def buscar(consulta, registros, matriz, k=3):
    """
    Busca los fragmentos más similares a una consulta mediante embeddings.

    Args:
        consulta: Texto del usuario que se desea buscar.
        registros: Fragmentos con sus metadatos.
        matriz: Matriz de embeddings de los fragmentos.
        k: Cantidad máxima de resultados a retornar.

    Returns:
        Lista de tuplas con cada fragmento recuperado y su puntaje de similitud.

    Raises:
        ValueError: Si la consulta está vacía o supera el límite de tokens
          del modelo.
    """
    # Validar consulta del usuario
    if not consulta.strip():
        raise ValueError("Escriba una consulta.")

    # Validar tokens de la consulta. Debe ser menor a 128
    if contar_tokens(consulta) > modelo.max_seq_length:
        raise ValueError("La consulta supera el límite del modelo.")

    # Convertir la consulta en embedding
    vector = modelo.encode(
        consulta,
        normalize_embeddings=True
    )

    # Calcular la similitud entre consulta y fragmento con producto matricial
    puntajes = matriz @ vector

    # Retornar los 3 resultados con el puntaje más alto en orden descendente
    return [
        (registros[int(i)], float(puntajes[i]))
        for i in np.argsort(-puntajes)[:k]
    ]

In [14]:
#@title Escriba aquí la consulta
CONSULTA = "¿Ante una inconsistencia quién prevalece, la Constitución o la ley?" #@param {type:"string"}

# Recorre las estrategias de fragmentación definidas en fragmentos
# es decir, ventanas y parrafos
for nombre in fragmentos:
    print("\nESTRATEGIA:", nombre)
    for registro, puntaje in buscar(
        CONSULTA,
        fragmentos[nombre], # Fragmentos de esa estrategia
        matrices[nombre] # Embeddings de esos fragmentos
    ):
        print(
            f"{registro['source']} · "
            f"{registro['section']} · "
            f"coseno {puntaje:.3f}")

        print(registro["chunk_id"])
        print(registro["texto"], "\n")


ESTRATEGIA: ventanas
constitucion_politica.md · Artículo 28 · coseno 0.601
constitucion_politica:1.0-implementacion:ventanas:33
CONSTITUCION POLITICA DE COLOMBIA / Artículo 28
Toda persona es libre. Nadie puede ser molestado en su persona o familia,
ni reducido a prisión o arresto, ni detenido, ni su domicilio registrado, sino en virtud de
mandamiento escrito de autoridad judicial competente, con las formalidades legales y
por motivo previamente definido en la ley.
La persona detenida preventivamente será puesta a disposición del juez competente
dentro de 

constitucion_politica.md · Artículo 29 · coseno 0.596
constitucion_politica:1.0-implementacion:ventanas:36
CONSTITUCION POLITICA DE COLOMBIA / Artículo 29
penal, la ley permisiva o favorable, aun cuando sea posterior, se aplicará de
preferencia a la restrictiva o desfavorable.
Toda persona se presume inocente mientras no se la haya declarado judicialmente
culpable. Quien sea sindicado tiene derecho a la defensa y a la asistencia de

Pruebar también «¿Cómo se hace para consultar el estado de un proceso?», «¿Cómo solicito una audiencia con el juez?». Antes de ejecutar, señalen dónde debería estar la evidencia. Para audiencia con juez no hay respuesta en este corpus, aunque la búsqueda devuelva candidatos. Anoten qué estrategia conserva mejor la información necesaria en cada caso.

## 6. Guardar y volver a cargar el índice
Elijan una estrategia después de comparar. Guardamos vectores, registros y configuración; luego los leemos nuevamente y repetimos la consulta. Esta colección pequeña utiliza una matriz para búsqueda exacta. En Colab descarguen el ZIP para conservarlo fuera de la sesión.

In [15]:
# Elegri la estrategia
ESTRATEGIA = "ventanas" #@param ["ventanas", "parrafos"]

# Crear el directorio de destino
destino = Path("indice_clase07")
destino.mkdir(exist_ok=True)

# Guardar configuración y parámetros
configuracion = {
    "modelo": MODELO,
    "revision": REVISION,
    "estrategia": ESTRATEGIA,
    "max_tokens": MAX_TOKENS,
    "solapamiento": SOLAPAMIENTO,
    "normalizados": True,
    "similitud": "coseno",
    "max_seq_length": modelo.max_seq_length,
}

# Guardar la matriz de embeddings
# Cada fila es el vector de un fragmento
# indice_clase07/vectores.npy
# Guardar fragmentos y la configuración
np.save(destino / "vectores.npy", matrices[ESTRATEGIA])
(destino / "registros.json").write_text(json.dumps(fragmentos[ESTRATEGIA], ensure_ascii=False, indent=2), encoding="utf-8")
(destino / "configuracion.json").write_text(json.dumps(configuracion, indent=2), encoding="utf-8")

# Leemos desde disco: no recalculamos los embeddings del corpus.
registros_cargados = json.loads((destino / "registros.json").read_text(encoding="utf-8"))
vectores_cargados = np.load(destino / "vectores.npy", allow_pickle=False)
config_cargada = json.loads((destino / "configuracion.json").read_text(encoding="utf-8"))

# Verificar mismo modelo y mismo número de vectores y registros
assert config_cargada["modelo"] == MODELO and vectores_cargados.shape[0] == len(registros_cargados)

# Buscar con los valores en memoria
antes = buscar(
    CONSULTA,
    fragmentos[ESTRATEGIA],
    matrices[ESTRATEGIA]
)

# Buscar con los valores cargados
despues = buscar(
    CONSULTA,
    registros_cargados,
    vectores_cargados
)

# Los resultados deben ser los mismos, listas iguales
assert [r["chunk_id"] for r, _ in antes] == [r["chunk_id"] for r, _ in despues]

# Los puntajes deben ser los mismos
assert np.allclose([p for _, p in antes], [p for _, p in despues])
print("Índice cargado: mismos fragmentos y puntajes para la consulta.")

# Crear archivo ZIP comprimido
with zipfile.ZipFile(
    "Clase_07_Indice_y_Corpus.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as archivo:
    for ruta in [*destino.iterdir(), *carpeta.glob("*.md")]:
        archivo.write(ruta, arcname=ruta.as_posix())

print("Archivo para conservar:", "Clase_07_Indice_y_Corpus.zip")

Índice cargado: mismos fragmentos y puntajes para la consulta.
Archivo para conservar: Clase_07_Indice_y_Corpus.zip


Para usar este índice en otra sesión: descompriman el ZIP, lean la configuración y carguen `SentenceTransformer(config_cargada["modelo"], revision=config_cargada["revision"])`. Lean los registros y la matriz como en la celda anterior. Las consultas deben usar esa misma representación.

In [16]:
#@title Descargar el índice y los documentos (opcional)
DESCARGAR = True #@param {type:"boolean"}
if DESCARGAR:
    from google.colab import files
    files.download("Clase_07_Indice_y_Corpus.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>